<a href="https://colab.research.google.com/github/lucasqueiros/hpc/blob/main/llm-lab/experiments/queiros_lorena/lorena_transformer_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import tiktoken

**Configurando DataLoader**

---

In [30]:
enc = tiktoken.get_encoding("gpt2")

In [31]:
from IPython.utils.path import target_outdated
class DataSet:
  def __init__(self,text ,tokenizer, context_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(text)

    for i in range(0, len(token_ids) - context_length, stride ):
      entrada = token_ids[i : i + context_length]
      target = token_ids[i + 1 : i + context_length + 1 ]

      self.input_ids.append(torch.tensor(entrada))
      self.target_ids.append(torch.tensor(target))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [32]:
def dataloader(text, batch_size=4, context_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = DataSet(text, tokenizer, context_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

**Implementando embbeding**

---

In [33]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text_verdict = f.read()

tokens_verdict = enc.encode(text_verdict)

In [34]:
vocab_size = 50257   # padrão GPT-2
embed_dim = 256
context_length = 256

embedding = nn.Embedding(vocab_size, embed_dim)

pos_embedding = nn.Embedding(context_length, embed_dim)



**implementando Self Attention**

---

In [35]:
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        self.embed_dim = embed_dim

        self.query = nn.Linear(embed_dim, embed_dim)
        self.key   = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        B, T, C = x.shape

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.embed_dim ** 0.5)

        mask = torch.tril(torch.ones(T, T, device=x.device))

        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)

        out = weights @ V

        return out

In [36]:
self_attn = SelfAttention(embed_dim)

# exemplo com batch
dl = dataloader(text_verdict)
x, y = next(iter(dl))

tokens = embedding(x)

positions = torch.arange(0, x.shape[1])
pos = pos_embedding(positions)

x_embed = tokens + pos

out = self_attn(x_embed)

print(out.shape)

torch.Size([4, 256, 256])


**implementando Multi Head AAttention**

---

In [37]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, - float('inf'))

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, -1)
        return self.out_proj(context_vec)

**Implementando Transformer**

---

In [38]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, context_length, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model, context_length, dropout, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

In [39]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, context_length, d_model, n_heads, n_layers, ffn_dim, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)

        self.drop_emb = nn.Dropout(dropout)

        self.transformer_blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, ffn_dim, context_length, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.out_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        tok_emb = self.token_emb(x)
        pos_ids = torch.arange(seq_len, device=x.device)
        pos_emb = self.pos_emb(pos_ids)

        x = self.drop_emb(tok_emb + pos_emb)

        x = self.transformer_blocks(x)
        x = self.ln_f(x)
        logits = self.out_head(x)
        return logits

In [40]:
model = MiniGPT(
    vocab_size = 50257,
    context_length = 256,
    d_model = 128,
    n_heads = 4,
    n_layers = 2,
    ffn_dim = 512
)

x = torch.randint(0, 50257, (2, 16))
logits = model(x)

print(logits.shape)

torch.Size([2, 16, 50257])


In [27]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

for d_model in [64, 128, 256]:
    print(f"d_model = {d_model}")

    model = MiniGPT(
        vocab_size = 50257,
        context_length = 256,
        d_model = d_model,
        n_heads = 4,
        n_layers = 2,
        ffn_dim = 4*d_model
    )

    x = torch.randint(0, 50257, (2, 16))
    out = model(x)
    print(f"Shape de saída: {out.shape}")

    total_params = count_parameters(model)
    print(f"Total de parâmetros: {total_params:,}")

d_model = 64
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 6,548,992
d_model = 128
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 13,294,592
d_model = 256
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 27,375,616


Quando aumentamos a dimensão do embbeding, os parâmetros aumentam também, o `d_model` afeta quase todas as camadas do modelo. As matrizes de peso das camadas lineares

In [28]:
for n_heads in [2, 4, 8]:
    print(f"n_heads = {n_heads}")

    model = MiniGPT(
        vocab_size = 50257,
        context_length = 256,
        d_model = 128,
        n_heads = n_heads,
        n_layers = 2,
        ffn_dim = 512
    )

    x = torch.randint(0, 50257, (2, 16))
    out = model(x)

    print(f"Shape de saída: {out.shape}")

    total_params = count_parameters(model)
    print(f"Total de parâmetros: {total_params:,}")

n_heads = 2
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 13,294,592
n_heads = 4
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 13,294,592
n_heads = 8
Shape de saída: torch.Size([2, 16, 50257])
Total de parâmetros: 13,294,592


O número de cabeças não adiciona novos pesos, ele apenas divide a dimensão total (d_model) em partes menores, mais cabeças permitem que o modelo foque em diferentes partes da frase simultaneamente, mas a quantidade de parâmetros usada pelas matrizes lineares permanece a mesma, então não altera isso.